In [ ]:
import pandas as pd

# ==============================
# FILE PATHS
# ==============================
mapping_file = r"D:/Tushar/main_with_subs_only.xlsx"
comparison_file = r"D:/Tushar/Comparison data.xlsx"

# ==============================
# READ FILES
# ==============================
mapping = pd.read_excel(mapping_file)
comparison = pd.read_excel(comparison_file)

mapping.columns = mapping.columns.str.strip()
comparison.columns = comparison.columns.str.strip()

# ==============================
# NORMALIZE KEYS
# ==============================
def normalize(series):
    return series.astype(str).str.strip().str.upper()

mapping['Sub_Label'] = normalize(mapping['Sub_Label'])
mapping['Main_Label'] = normalize(mapping['Main_Label'])
comparison['Material'] = normalize(comparison['Material'])

mapping['Sub_Count'] = pd.to_numeric(mapping['Sub_Count'], errors='coerce').fillna(0)

# ==============================
# MERGE SWITCH WITH BOM
# ==============================
merged = mapping.merge(
    comparison,
    left_on='Sub_Label',
    right_on='Material',
    how='left'
)

# ==============================
# IDENTIFY DATE COLUMNS
# ==============================
indent_cols = [c for c in merged.columns if 'Indent' in c]
actual_cols = [c for c in merged.columns if 'Production Plan' in c]

# Ensure correct order
indent_cols.sort()
actual_cols.sort()

results = []

# ==============================
# LOOP THROUGH EACH DAY
# ==============================
for indent_col, actual_col in zip(indent_cols, actual_cols):

    date = indent_col.split()[0]

    merged['Tentative'] = pd.to_numeric(merged[indent_col], errors='coerce').fillna(0)
    merged['Actual'] = pd.to_numeric(merged[actual_col], errors='coerce').fillna(0)

    merged['Tentative_Child'] = merged['Tentative'] * merged['Sub_Count']
    merged['Actual_Child'] = merged['Actual'] * merged['Sub_Count']

    daily = merged.groupby('Main_Label').agg({
        'Actual_Child': 'sum',
        'Tentative_Child': 'sum'
    }).reset_index()

    daily['Date'] = date

    results.append(daily)

# ==============================
# COMBINE ALL DAYS
# ==============================
final = pd.concat(results, ignore_index=True)

# Reorder columns
final = final[['Date', 'Main_Label', 'Actual_Child', 'Tentative_Child']]

final.rename(columns={
    'Main_Label': 'Child_Part',
    'Actual_Child': 'Daily_Actual_Consumption',
    'Tentative_Child': 'Daily_Tentative_Requirement'
}, inplace=True)

print(final.head(20))

# ==============================
# SAVE OUTPUT
# ==============================
output_file = "Daily_Child_Comparison.xlsx"
final.to_excel(output_file, index=False)

print("\nSaved to:", output_file)
